# Hindi 25M LM - reasoning finetuning only (rerun with the improved dataset)
Settings: **GPU T4**, **Internet On**. Inputs: the `hindi-tokens` dataset + the **pretraining notebook's output** (checkpoint).
Writes results to `/kaggle/working/report/` (accuracy json, predictions, attention figures).

In [ ]:
import subprocess, os, glob, shutil, json
if not os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git','clone','https://github.com/Keerthi-2012/Monolingual-Transformer-LM.git','/kaggle/working/repo'],check=True)
else:
    subprocess.run(['git','-C','/kaggle/working/repo','pull'],check=True)
print(subprocess.run(['git','-C','/kaggle/working/repo','log','-1','--oneline'],capture_output=True,text=True).stdout)
!apt-get -qq install -y fonts-noto-core > /dev/null 2>&1; fc-cache -f > /dev/null 2>&1
TOKENS = os.path.dirname(glob.glob('/kaggle/input/**/train_000.bin', recursive=True)[0])
CK = sorted(glob.glob('/kaggle/input/**/ckpt_best.pt', recursive=True), key=os.path.getmtime)[-1]
print('tokens:', TOKENS); print('checkpoint:', CK)
R = '/kaggle/working/report'; os.makedirs(f'{R}/attn', exist_ok=True); os.makedirs(f'{R}/preds', exist_ok=True)

In [ ]:
# the reasoning dataset is deterministic and not stored in git: generate it here (a few seconds)
!cd /kaggle/working/repo/hindi/reasoning && python generate.py --out data

## Zero-shot baseline -> finetune -> evaluate (also prints a name-copy diagnostic)

In [ ]:
!cd /kaggle/working/repo/hindi/reasoning && python -u finetune.py --pretrained {CK} --out-dir /kaggle/working/runs/ft

In [ ]:
FT = '/kaggle/working/runs/ft'
for f in glob.glob(f'{FT}/acc_*.json') + [f'{FT}/finetune_log.jsonl']: shutil.copy(f, R)
for f in glob.glob(f'{FT}/preds_*.jsonl'): shutil.copy(f, f'{R}/preds')
for f in ('acc_pretrained.json','acc_finetuned.json'):
    print('=====', f); print(json.dumps(json.load(open(f'{R}/{f}')), indent=1))
# a few predictions to read
for l in open(f'{R}/preds/preds_finetuned_test_iid.jsonl').read().splitlines()[:6]:
    r = json.loads(l); print(r['qtype'], r['ok'], '| gold:', r['gold'], '| pred:', r['pred'][:200].replace(chr(10), ' / '))

## Attention analysis (pretrained and finetuned, same toolkit)

In [ ]:
!cd /kaggle/working/repo/hindi/eval && python attn_analysis.py --ckpt {CK} --tokens {TOKENS} --out-dir {R}/attn --name pretrained --reasoning-data /kaggle/working/repo/hindi/reasoning/data/test_iid.jsonl
!cd /kaggle/working/repo/hindi/eval && python attn_analysis.py --ckpt {FT}/ckpt_finetuned.pt --tokens {TOKENS} --out-dir {R}/attn --name finetuned --reasoning-data /kaggle/working/repo/hindi/reasoning/data/test_iid.jsonl